# <font color="#418FDE" size="10" uppercase>**C: ML Major Models**</font>
----

> Last update: 20231210

By the end of this lecture, you will be able to:
* Investigate and apply conventional, popular, and memory-efficient ML architectures.
* Describe segmentation, object detection, generative, and multi-modal ML models.

## **1. Conventional Models**

Conventional NN models like LeNet & AlexNet mark significant milestones in the field of deep learning & have laid the groundwork for modern advancements in this area. LeNet, developed by Yann LeCun in the late 1980s, is one of the earliest convolutional NNs (CNNs) & was primarily designed for handwritten digit recognition. It introduced key concepts like convolutional layers & pooling layers, which are now standard in CNN architectures. LeNet's structure, consisting of alternating convolutional & pooling layers followed by fully connected layers, set a foundational template for many subsequent NN designs. On the other hand, AlexNet, developed by Alex Krizhevsky, Ilya Sutskever, & Geoffrey Hinton in 2012, was a breakthrough model that significantly outperformed other competitors in the ImageNet Large Scale Visual Recognition Challenge (ILSVRC). It featured deeper layers & an increased number of parameters compared to previous models, along with innovative techniques like ReLU (Rectified Linear Unit) activation, dropout for reducing overfitting, & the use of GPU computing for training. AlexNet's success effectively triggered the deep learning revolution in the field of computer vision, demonstrating the potential of deep, large-scale NNs in solving complex perceptual tasks. Both LeNet & AlexNet not only proved the efficacy of NNs in image recognition tasks but also inspired an array of subsequent research & development in deep learning architectures.

### **1.1 LeNet**


> [LeNet](http://vision.stanford.edu/cs598_spring07/papers/Lecun98.pdf), developed by Yann LeCun in 1998, is a pioneering convolutional NN (CNN) that played a crucial role in the development of deep learning. It was specifically designed for handwriting recognition & digit classification tasks, particularly for recognizing characters in documents & postal codes in mail sorting systems. LeNet's architecture is relatively simple by modern standards, yet it laid the foundation for more complex & sophisticated CNNs.

> The network consists of several layers, including convolutional layers, pooling (subsampling) layers, & fully connected layers. The convolutional layers are responsible for extracting features from input images through learned filters, while the pooling layers reduce the spatial size of the representations, thereby decreasing the number of parameters & computation required in the network. This process helps in achieving translational invariance in the recognition process. The fully connected layers, towards the end of the network, perform classification based on the features extracted & pooled in the previous layers.

> LeNet was one of the first successful applications of CNNs & demonstrated the potential of deep learning for practical image recognition tasks. Its architecture has inspired numerous advancements in deep learning & computer vision, setting the stage for the development of more complex networks like AlexNet, VGGNet, & others.

In [ ]:
#@title Example of LeNet (Lightly Commented)
'''
Runtime: GPU $$$
tf.keras.datasets.mnist:              https://www.tensorflow.org/api_docs/python/tf/keras/datasets/mnist
tf.keras.utils.to_categorical:        https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical
tf.data.Dataset.from_tensor_slices:   https://www.tensorflow.org/api_docs/python/tf/data/Dataset#from_tensor_slices
tf.data.Dataset.shuffle:              https://www.tensorflow.org/api_docs/python/tf/data/Dataset#shuffle
tf.data.Dataset.batch:                https://www.tensorflow.org/api_docs/python/tf/data/Dataset#batch
tf.data.Dataset.prefetch:             https://www.tensorflow.org/api_docs/python/tf/data/Dataset#prefetch
tf.keras.Input:                       https://www.tensorflow.org/api_docs/python/tf/keras/Input
tf.keras.layers.Conv2D:               https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D
tf.keras.layers.MaxPooling2D:         https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPooling2D
tf.keras.layers.Flatten:              https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten
tf.keras.layers.Dense:                https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.Model:                       https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.optimizers.Adam:                   https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam
tf.keras.callbacks.EarlyStopping:     https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
tf.keras.callbacks.ReduceLROnPlateau: https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau
'''

import tensorflow as tf

# Load MNIST dataset (handwritten digits images & labels)
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.mnist.load_data()

# Reshape & normalize training & validation data for CNN compatibility
datain_tr = datain_tr.reshape(datain_tr.shape[0], 28, 28, 1).astype('float32') / 255
datain_vl = datain_vl.reshape(datain_vl.shape[0], 28, 28, 1).astype('float32') / 255

# One-hot encode the labels for training & validation data
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

# Function to create a TensorFlow Dataset for efficient data handling
def fun_create_dataset(data, labels, batch_size=32, train=True):
    dataset = tf.data.Dataset.from_tensor_slices((data, labels))
    if train: dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size)
    return dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

# Create TensorFlow Datasets for training & validation
data_tr = fun_create_dataset(datain_tr, dataou_tr, batch_size=128, train=True)
data_vl = fun_create_dataset(datain_vl, dataou_vl, batch_size=128, train=False)

# Define the LeNet model architecture using the functional API
inputs = tf.keras.Input(shape=(28, 28, 1))
x = tf.keras.layers.Conv2D(6, (5, 5), activation='relu')(inputs)
x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
x = tf.keras.layers.Conv2D(16, (5, 5), activation='relu')(x)
x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(120, activation='relu')(x)
x = tf.keras.layers.Dense(84, activation='relu')(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
model = tf.keras.Model(inputs=inputs, outputs=outputs, name="model")

# Compile the model with optimizer, loss function, & metrics
model.compile(optimizer=tf.optimizers.Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()

# Set up callbacks for early stopping & learning rate reduction
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=0.00001)

# Train the model
history = model.fit(data_tr, epochs=100, validation_data=data_vl, callbacks=[early_stopping, reduce_lr])

# Evaluate the model's performance on validation data
test_loss, test_accuracy = model.evaluate(data_vl)

# Print test loss & accuracy
print(f'Test Loss: {test_loss}, Test Accuracy: {test_accuracy}')

# Auto runtime disconnection to save CPU/GPU/TPU allocations
try:
  from google.colab import runtime
  import time
  time.sleep(5)
  runtime.unassign()
except ImportError:
  pass

In [ ]:
#@title Example of LeNet (Fully Commented)
'''
Runtime: GPU $$$
tf.keras.datasets.mnist:              https://www.tensorflow.org/api_docs/python/tf/keras/datasets/mnist
tf.keras.utils.to_categorical:        https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical
tf.data.Dataset.from_tensor_slices:   https://www.tensorflow.org/api_docs/python/tf/data/Dataset#from_tensor_slices
tf.data.Dataset.shuffle:              https://www.tensorflow.org/api_docs/python/tf/data/Dataset#shuffle
tf.data.Dataset.batch:                https://www.tensorflow.org/api_docs/python/tf/data/Dataset#batch
tf.data.Dataset.prefetch:             https://www.tensorflow.org/api_docs/python/tf/data/Dataset#prefetch
tf.keras.Input:                       https://www.tensorflow.org/api_docs/python/tf/keras/Input
tf.keras.layers.Conv2D:               https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D
tf.keras.layers.MaxPooling2D:         https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPooling2D
tf.keras.layers.Flatten:              https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten
tf.keras.layers.Dense:                https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.Model:                       https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.optimizers.Adam:                   https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam
tf.keras.callbacks.EarlyStopping:     https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
tf.keras.callbacks.ReduceLROnPlateau: https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau
'''

import tensorflow as tf

# Load & preprocess the MNIST dataset
# Load the MNIST dataset using TensorFlow's built-in function.
# The MNIST dataset consists of 28x28 pixel handwritten digit images & their labels.
# It is split into a training set & a validation (or test) set.
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.mnist.load_data()

# Reshape the training data for compatibility with the model.
# Original data shape is (number_of_samples, 28, 28). Each image is 28x28 pixels.
# We reshape it to (number_of_samples, 28, 28, 1) to add a channel dimension, making it
# suitable for CNN models which expect this shape. The '1' signifies a single channel (grayscale).
datain_tr = datain_tr.reshape(datain_tr.shape[0], 28, 28, 1).astype('float32') / 255

# Reshape the validation data similarly to the training data.
datain_vl = datain_vl.reshape(datain_vl.shape[0], 28, 28, 1).astype('float32') / 255

# Normalize the pixel values of the images.
# Originally, pixel values are integers in the range [0, 255].
# We normalize these to be in the range [0, 1] which aids in training NNs.
datain_tr /= 255
datain_vl /= 255

# Convert the labels from integer to categorical (one-hot encoding).
# MNIST labels are integers from 0 to 9, representing the digit in the image.
# One-hot encoding converts these into a binary matrix representation which is needed
# for categorical cross entropy loss in classification problems.
# For example, the label '3' will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0].
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

# Create TensorFlow Datasets
def fun_create_dataset(data, labels, batch_size=32, train=True):
    # Create a tf.data.Dataset object from the given data & labels.
    # This method is a way to slice the array of data & labels pair-wise.
    # It creates a dataset that yields each element of the array as a tuple.
    dataset = tf.data.Dataset.from_tensor_slices((data, labels))

    # If the 'train' flag is set to True, shuffle the dataset.
    # 'buffer_size=1024' means that 1024 elements will be loaded into buffer & shuffled.
    # Shuffling is important for training data to ensure the model does not learn
    # the order of the training data. The larger the buffer, the better the shuffle,
    # but it requires more memory to hold the elements in the buffer.
    if train: dataset = dataset.shuffle(buffer_size=1024)

    # Batch the data into the specified batch size.
    # This means that the dataset will yield batches of data, each containing a
    # specified number of data points (defined by 'batch_size').
    # Batching is necessary for training in manageable sized chunks.
    dataset = dataset.batch(batch_size)

    # Prefetch elements from the dataset ahead of time.
    # 'buffer_size=tf.data.experimental.AUTOTUNE' allows TensorFlow to automatically
    # manage the buffer size based on available resources & dataset characteristics.
    # Prefetching is used to reduce the time the model spends waiting for data to
    # be loaded, thereby speeding up training.
    return dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

# Our GPU has enough memory to handle a large batch size for this problem.
data_tr = fun_create_dataset(datain_tr, dataou_tr, batch_size=128, train=True)
data_vl = fun_create_dataset(datain_tr, dataou_tr, batch_size=128, train=False)

# Define the LeNet model using the functional API

# Define the input layer of the model.
# This specifies the shape of the input data which the model will accept.
# Here, the shape (28, 28, 1) typically represents a 28x28 pixel grayscale image (1 channel).
inputs = tf.keras.Input(shape=(28, 28, 1))

# Add the first convolutional layer.
# This layer has 6 filters (or kernels), each with a size of 5x5, & uses the ReLU activation function.
# Convolutional layers are used to extract features from the input image.
x = tf.keras.layers.Conv2D(6, (5, 5), activation='relu')(inputs)

# Add the first max pooling layer.
# This layer reduces the spatial dimensions (width & height) of the output from the previous layer.
# It helps in reducing the computational load, memory usage, & also helps in reducing overfitting.
# Here, it uses a pooling window of size 2x2.
x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)

# Add the second convolutional layer with 16 filters, each of size 5x5.
# The ReLU activation function is again used.
x = tf.keras.layers.Conv2D(16, (5, 5), activation='relu')(x)

# Add the second max pooling layer, also with a pooling window of size 2x2.
x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)

# Flatten the output from the previous layer.
# This is necessary to transition from 2D feature maps to 1D feature vectors for the dense layers.
x = tf.keras.layers.Flatten()(x)

# Add a dense (fully connected) layer with 120 units (or neurons) & ReLU activation.
# Dense layers are used for learning non-linear combinations of the high-level features extracted by the convolutional layers.
x = tf.keras.layers.Dense(120, activation='relu')(x)

# Add another dense layer with 84 units & ReLU activation.
x = tf.keras.layers.Dense(84, activation='relu')(x)

# Define the output layer of the model.
# This dense layer has 10 units, corresponding to 10 classes (for a 10-class classification problem).
# The softmax activation function is used to output a probability distribution over the 10 classes.
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)

# Create the model by specifying the inputs & outputs.
# 'inputs' refers to the input layer defined at the start.
# 'outputs' refers to the output of the final layer defined above.
# The model is named "model".
model = tf.keras.Model(inputs=inputs, outputs=outputs, name="model")


# Compile the model
model.compile(optimizer=tf.optimizers.Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

# Compile the model with an optimizer, loss function, & evaluation metrics.
model.compile(
    optimizer='adam',
    # The optimizer 'adam' is used for training the model.
    # Adam is an optimization algorithm that can be used instead of the classical stochastic gradient descent procedure
    # to update network weights iteratively based on training data.
    # Adam is known for being effective & efficient & is a good default choice for a wide range of problems.

    loss='categorical_crossentropy',
    # The loss function is set to 'categorical_crossentropy', which is a common choice for classification problems.
    # This loss function is used when there are two or more label classes.
    # It expects labels to be provided in a one-hot encoded format.
    # This loss function measures the performance of the model whose output is a probability value between 0 & 1.

    metrics=['accuracy']
    # The metric for evaluating the model is set to 'accuracy'.
    # Accuracy calculates how often predictions equal labels.
    # It is a common metric for classification problems to understand the model's performance in simple terms.
    # This will allow us to monitor the percentage of correctly classified instances during training & testing.
)


# Model summary
model.summary()

# Create an EarlyStopping callback instance.
# This callback will monitor the validation loss ('val_loss') during training.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',  # The metric to be monitored. In this case, it's the validation loss.
    patience=3,          # Number of epochs with no improvement after which training will be stopped.
                         # Here, it stops training if 'val_loss' does not improve for 3 consecutive epochs.
    restore_best_weights=True  # Whether to restore model weights from the epoch with the best value of the monitored quantity.
                               # If True, the model weights obtained at the end of the best epoch are used.
)

# Create a ReduceLROnPlateau callback instance.
# This callback reduces the learning rate when a metric has stopped improving.
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',  # The metric to be monitored. Again, this is the validation loss.
    factor=0.5,          # Factor by which the learning rate will be reduced. new_lr = lr * factor.
                         # Here, the new learning rate will be 20% of the current learning rate.
    patience=2,          # Number of epochs with no improvement after which learning rate will be reduced.
                         # Here, if 'val_loss' does not improve for 2 consecutive epochs, the learning rate is reduced.
    min_lr=0.00001       # Lower bound on the learning rate.
                         # The learning rate will not be reduced below this value.
)


# Train the model with the provided training data
history = model.fit(
    data_tr,  # Training data. The variable 'data_tr' should be a tf.data.Dataset or a tuple of numpy arrays.
              # It represents the input features & corresponding target labels for training the model.

    epochs=100, # Number of epochs to train the model. An epoch is an iteration over the entire training data.
                # Here, the model will go through the training data 10 times.

    validation_data=data_vl,  # Validation data. The variable 'data_vl' is similar to 'data_tr' but is used
                              # for evaluating the model's performance after each epoch. This helps in monitoring
                              # the model's performance on unseen data & in preventing overfitting.

    callbacks=[early_stopping, reduce_lr]  # List of callbacks to apply during training.
                                           # 'early_stopping' will stop training early if the validation loss doesn't improve.
                                           # 'reduce_lr' will reduce the learning rate if the validation loss stops improving.
)


# Evaluate the trained model using the test dataset.
# The 'model.evaluate' function tests the model's performance on a given dataset.
# It returns the loss value & metrics values (like accuracy) for the model in test mode.
# 'data_vl' should be the dataset reserved for testing. It usually contains input data
# & its corresponding labels. It is not used during the training phase & helps
# in assessing the model's generalization capability.
test_loss, test_accuracy = model.evaluate(data_vl)

# Print the evaluation results.
# After evaluating the model, we obtain two main metrics: test_loss & test_accuracy.
# 'test_loss' indicates how well the model is doing in terms of the loss function it's using.
# Lower loss values are generally better, indicating that the model's predictions are
# closer to the true labels.

# 'test_accuracy' is a measure of how often the model's predictions match the true labels.
# It is given as a fraction (between 0 & 1), where higher values indicate better performance.
# In many cases, especially for classification tasks, accuracy is a key metric.

# The f-string in Python is used here for formatted string literals.
# It's a convenient way to embed expressions inside string literals for formatting.
print(f'Test Loss: {test_loss}, Test Accuracy: {test_accuracy}')

# Auto runtime disconnection to save CPU/GPU/TPU allocations
try:
  from google.colab import runtime
  import time
  time.sleep(5)
  runtime.unassign()
except ImportError:
  pass

### **1.2. AlexNet**

> [AlexNet](https://proceedings.neurips.cc/paper/4824-imagenet-classification-with-deep-convolutional-neural-networks.pdf), introduced in 2012 by Alex Krizhevsky, Ilya Sutskever, & Geoffrey Hinton, is a seminal architecture in the field of deep learning, particularly in computer vision. This NN marked a breakthrough in the ImageNet Large Scale Visual Recognition Challenge (ILSVRC), where it significantly outperformed other competitors, reducing the top-5 error rate by a substantial margin. AlexNet's design is a deeper & wider version of LeNet & was the first to use Rectified Linear Units (ReLU) for the activation functions, which helped to speed up the training process considerably.

> The architecture of AlexNet consists of five convolutional layers, some of which are followed by max-pooling layers, & three fully connected layers at the end. One of the key innovations of AlexNet was the use of dropout layers, a technique to prevent overfitting in the fully connected layers. AlexNet also implemented overlapping pooling, a strategy to reduce the size of the network & improve its performance & generalization. The network was trained on two GPUs, which was a novelty at the time, & this approach helped manage the computational load.

> AlexNet's success in the ImageNet challenge was a turning point for the field of deep learning, showcasing the capabilities of NNs in handling large-scale image recognition tasks. Its success spurred a wave of interest & research in deep learning, leading to the rapid development of more advanced NN architectures.

In [ ]:
#@title Example of AlexNet
'''
Runtime: GPU $$$
tf.keras.datasets.cifar10:            https://www.tensorflow.org/api_docs/python/tf/keras/datasets/cifar10
tf.keras.utils.to_categorical:        https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical
tf.keras.Input:                       https://www.tensorflow.org/api_docs/python/tf/keras/Input
tf.keras.layers.Resizing:             https://www.tensorflow.org/api_docs/python/tf/keras/layers/Resizing
tf.keras.layers.Conv2D:               https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D
tf.keras.layers.MaxPooling2D:         https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPooling2D
tf.keras.layers.Flatten:              https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten
tf.keras.layers.Dense:                https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.layers.Dropout:              https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout
tf.keras.Model:                       https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.optimizers.Adam:                   https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam
tf.keras.callbacks.EarlyStopping:     https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
tf.keras.callbacks.ReduceLROnPlateau: https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau

'''

import tensorflow as tf

# Load CIFAR10 dataset
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.cifar10.load_data()

# Normalize the data
datain_tr = datain_tr/ 255
datain_vl = datain_vl / 255

# Convert labels to one-hot encoding
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

# Function to create a TensorFlow Dataset for efficient data handling
def fun_create_dataset(data, labels, batch_size=32, train=True):
    dataset = tf.data.Dataset.from_tensor_slices((data, labels))
    if train:
        dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size)
    return dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

data_tr = fun_create_dataset(datain_tr, dataou_tr, batch_size=64, train=True)
data_vl = fun_create_dataset(datain_vl, dataou_vl, batch_size=64, train=False)

# Define the AlexNet model
inputs = tf.keras.Input(shape=(32, 32, 3))
# Resize images to the size AlexNet expects (224x224)
x = tf.keras.layers.Resizing(224, 224, interpolation='bilinear')(inputs)
x = tf.keras.layers.Conv2D(filters=96, kernel_size=(11, 11), strides=4, activation='relu')(x)
x = tf.keras.layers.MaxPooling2D(pool_size=(3, 3), strides=2)(x)
x = tf.keras.layers.Conv2D(filters=256, kernel_size=(5, 5), padding='same', activation='relu')(x)
x = tf.keras.layers.MaxPooling2D(pool_size=(3, 3), strides=2)(x)
x = tf.keras.layers.Conv2D(filters=384, kernel_size=(3, 3), padding='same', activation='relu')(x)
x = tf.keras.layers.Conv2D(filters=384, kernel_size=(3, 3), padding='same', activation='relu')(x)
x = tf.keras.layers.Conv2D(filters=256, kernel_size=(3, 3), padding='same', activation='relu')(x)
x = tf.keras.layers.MaxPooling2D(pool_size=(3, 3), strides=2)(x)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(4096, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
x = tf.keras.layers.Dense(4096, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name='model')

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()

# Set up callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=0.00001)

# Train the model
history = model.fit(data_tr, epochs=2, validation_data=data_vl, callbacks=[early_stopping, reduce_lr])

# Evaluate the model
test_loss, test_accuracy = model.evaluate(data_vl)

# Print test results
print(f'Test Loss: {test_loss}, Test Accuracy: {test_accuracy}')


# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

## **2. Keras Applications**


[Keras Applications](https://keras.io/api/applications/) is a module within the Keras deep learning library that provides pre-trained models. These models are trained on large datasets like ImageNet & can be used for a variety of tasks including image classification, feature extraction, & fine-tuning for specific use cases. The pre-trained models in Keras Applications include well-known architectures such as ResNet, VGG, Inception, & MobileNet, among others. These models vary in size, complexity, & performance, offering a range of options depending on the specific requirements of the task at hand. For instance, models like MobileNet are designed for efficiency, making them suitable for mobile or edge computing, while others like ResNet are designed for high performance in more resource-intensive applications. Utilizing these pre-trained models can significantly save time & resources, as training such models from scratch requires substantial computational power & a large amount of data. Additionally, Keras Applications makes it easy to customize these models for specific tasks by allowing additional layers to be added or existing layers to be fine-tuned, thereby adapting the model to new tasks with relatively less data compared to training a model from scratch.

### **2.1. VGGNets**

> [VGGNets](https://arxiv.org/abs/1409.1556), developed by Karen Simonyan & Andrew Zisserman of the Visual Graphics Group at the University of Oxford (hence the name VGG), represent a significant advancement in the field of deep learning, particularly in computer vision. Introduced in 2014, VGGNet models, especially [VGG-16 & VGG-19](https://www.kaggle.com/code/blurredmachine/vggnet-16-architecture-a-complete-guide?scriptVersionId=39674893&cellId=6), are known for their deep architectures & their success in the ImageNet Large Scale Visual Recognition Challenge (ILSVRC). The core idea behind VGGNets is the use of very small (3x3) convolutional filters throughout the entire network, which allowed for the construction of deeper networks.

> The key innovation of VGGNets lies in their simplicity & depth. They consist of 16 to 19 layers (in VGG-16 & VGG-19, respectively) of convolutional layers, each followed by a ReLU activation function, interspersed with max pooling layers. The use of small convolutional filters, as opposed to larger ones in previous architectures, enabled the stacking of more convolutional layers while controlling the number of parameters, thus increasing the depth of the network without a proportional increase in computational complexity. This approach allows VGGNets to learn more complex features at various levels of abstraction, which contributes to their strong performance in image classification tasks.

> Another notable aspect of VGGNets is their uniform architecture, which, despite its simplicity, proved to be highly effective & influential. The architecture inspired many subsequent studies & applications in deep learning, setting a precedent for the design of deep convolutional NNs. Furthermore, the pre-trained VGG models have been widely used as feature extractors for various computer vision tasks beyond classification, such as in object detection & image segmentation.

> Despite their impressive performance, VGGNets are also known for their large size & high computational cost, which can make them impractical for deployment in environments with limited computational resources. Nonetheless, the VGGNets remain a foundational model in the deep learning & computer vision community, illustrating the benefits of depth & simplicity in NN architecture.

In [ ]:
#@title Example of VGG16
'''
Runtime: GPU $$$
TensorFlow Standard Models:             https://keras.io/api/applications/
tf.keras.applications.VGG16:            https://www.tensorflow.org/api_docs/python/tf/keras/applications/VGG16
tf.keras.datasets.cifar10:              https://www.tensorflow.org/api_docs/python/tf/keras/datasets/cifar10
tf.keras.utils.to_categorical:          https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical
tf.data.Dataset.from_tensor_slices:     https://www.tensorflow.org/api_docs/python/tf/data/Dataset#from_tensor_slices
tf.data.Dataset.shuffle:                https://www.tensorflow.org/api_docs/python/tf/data/Dataset#shuffle
tf.data.Dataset.batch:                  https://www.tensorflow.org/api_docs/python/tf/data/Dataset#batch
tf.data.Dataset.prefetch:               https://www.tensorflow.org/api_docs/python/tf/data/Dataset#prefetch
tf.keras.Input:                         https://www.tensorflow.org/api_docs/python/tf/keras/Input
tf.keras.layers.GlobalAveragePooling2D: https://www.tensorflow.org/api_docs/python/tf/keras/layers/GlobalAveragePooling2D
tf.keras.layers.Dense:                  https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.layers.Dropout:                https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout
tf.keras.Model:                         https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.keras.callbacks.EarlyStopping:       https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
tf.keras.callbacks.ReduceLROnPlateau:   https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau
'''
import tensorflow as tf

# Load & preprocess CIFAR-10 dataset
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.cifar10.load_data()
datain_tr, datain_vl = datain_tr / 255.0, datain_vl / 255.0
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

def fun_create_dataset(data, labels, batch_size=32, train=True):
    dataset = tf.data.Dataset.from_tensor_slices((data, labels))
    if train: dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size)
    return dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

# Create training & validation datasets
data_tr = fun_create_dataset(datain_tr, dataou_tr, batch_size=32, train=True)
data_vl = fun_create_dataset(datain_vl, dataou_vl, batch_size=32, train=False)

# Load the base model with ImageNet weights
base_model = tf.keras.applications.VGG16(include_top=False,
                                         weights='imagenet',
                                         input_shape=(32, 32, 3))
# Base model summary
print("\n\n\n")
print("Summary of Base Model")
print("\n\n\n")
base_model.summary()

# Create the model
inputs  = tf.keras.Input(shape=(32, 32, 3))
x       = base_model(inputs)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dense(512, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
model   = tf.keras.Model(inputs=inputs, outputs=outputs, name='model')

# Set up callbacks for early stopping & learning rate reduction
cb_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                     patience=3,
                                                     restore_best_weights=True)

cb_reduce_lr      = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
                                                         factor=0.5,
                                                         patience=2,
                                                         min_lr=0.00001)

'''
Transfer Learning
'''

# Freeze the convolutional base
base_model.trainable = False

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
print("\n\n\n")
print("Summary of Final Model for Transfer Learning")
print("\n\n\n")
model.summary()

history = model.fit(data_tr,
                    epochs=3,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

'''
Fine-Tuning
'''

# Freeze the convolutional base
base_model.trainable = True

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.00001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
print("\n\n\n")
print("Summary of Final Model for Fine Tuning")
print("\n\n\n")
model.summary()

history = model.fit(data_tr,
                    epochs=2,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

# Evaluate the model
loss_vl, accuracy_vl = model.evaluate(data_vl)
print("\n\n\n")
print(f'Test Loss: {loss_vl}, Test Accuracy: {accuracy_vl}')
print("\n\n\n")

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **2.2. GoogLeNet**

> [GoogLeNet](https://arxiv.org/abs/1409.4842) also known as Inception v1, is a convolutional NN architecture that was introduced by researchers at Google in 2014. The primary innovation of GoogLeNet is its inception module, which allows the network to choose from multiple convolutional operations with different filter sizes & pooling operations in the same layer. This approach enables the model to capture information at various scales & complexities, making it highly effective for image recognition tasks. GoogLeNet was designed to optimize both accuracy & computational efficiency, addressing the problem of computational resource constraints while improving performance. This was particularly important for deploying models in real-world applications where resources like memory & processing power are limited.

> In terms of architecture, GoogLeNet is significantly deeper than previous models, but it is carefully designed to reduce the number of parameters & computational cost. This is achieved through the use of 1x1 convolutions to reduce dimensionality & the careful arrangement of the inception modules. The result is a deep network that offers state-of-the-art performance on image recognition tasks while being more efficient in terms of computation compared to its predecessors. GoogLeNet also introduced auxiliary classifiers to mitigate the vanishing gradient problem during training, enabling effective training of deeper network layers. These innovations contributed to GoogLeNet winning the ImageNet Large Scale Visual Recognition Challenge (ILSVRC) in 2014, setting a new benchmark in image classification tasks.

> However, GoogLeNet does have its drawbacks. The complexity of the inception module, while beneficial for performance, can make the network architecture more challenging to understand & modify compared to more straightforward designs like AlexNet or VGGNet. This complexity can also pose challenges in the implementation & optimization of the model. Additionally, while GoogLeNet reduces the computational burden compared to some other models, it still requires substantial computational resources to train, which may not be feasible for individuals or organizations with limited hardware capabilities. Despite these challenges, GoogLeNet's innovative approach to convolutional NN design has had a lasting impact on the field of computer vision, inspiring subsequent iterations & improvements in the Inception family of models.

In [ ]:
#@title Example of GoogleNet
'''
Runtime: GPU $$$
TensorFlow Standard Models:             https://keras.io/api/applications/
tf.keras.applications.InceptionV3:      https://www.tensorflow.org/api_docs/python/tf/keras/applications/InceptionV3
tf.keras.datasets.cifar10:              https://www.tensorflow.org/api_docs/python/tf/keras/datasets/cifar10
tf.keras.utils.to_categorical:          https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical
tf.data.Dataset.from_tensor_slices:     https://www.tensorflow.org/api_docs/python/tf/data/Dataset#from_tensor_slices
tf.data.Dataset.shuffle:                https://www.tensorflow.org/api_docs/python/tf/data/Dataset#shuffle
tf.data.Dataset.batch:                  https://www.tensorflow.org/api_docs/python/tf/data/Dataset#batch
tf.data.Dataset.prefetch:               https://www.tensorflow.org/api_docs/python/tf/data/Dataset#prefetch
tf.keras.Input:                         https://www.tensorflow.org/api_docs/python/tf/keras/Input
tf.keras.layers.GlobalAveragePooling2D: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten
tf.keras.layers.Dense:                  https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.layers.Dropout:                https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout
tf.keras.Model:                         https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.keras.callbacks.EarlyStopping:       https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
tf.keras.callbacks.ReduceLROnPlateau:   https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau
'''
import tensorflow as tf

# Load & preprocess CIFAR-10 dataset
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.cifar10.load_data()
datain_tr, datain_vl = datain_tr / 255.0, datain_vl / 255.0
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

def fun_create_dataset(data, labels, batch_size=32, train=True):
    dataset = tf.data.Dataset.from_tensor_slices((data, labels))
    if train: dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size)
    return dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

# Create training & validation datasets
data_tr = fun_create_dataset(datain_tr, dataou_tr, batch_size=32, train=True)
data_vl = fun_create_dataset(datain_vl, dataou_vl, batch_size=32, train=False)

# Load the base model with ImageNet weights
base_model = tf.keras.applications.InceptionV3(include_top=False,
                                               weights='imagenet',
                                               input_shape=(75, 75, 3))

# Base model summary
print("\n\n\n")
print("Summary of Base Model")
print("\n\n\n")
base_model.summary()

# Create the model
inputs  = tf.keras.Input(shape=(32, 32, 3))
x       = tf.keras.layers.Resizing(75,75)(inputs)
x       = base_model(x)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dense(1024, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
model   = tf.keras.Model(inputs=inputs, outputs=outputs, name='model')

# Set up callbacks for early stopping & learning rate reduction
cb_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                     patience=3,
                                                     restore_best_weights=True)

cb_reduce_lr      = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
                                                         factor=0.5,
                                                         patience=2,
                                                         min_lr=0.00001)

'''
Transfer Learning
'''

# Freeze the convolutional base
base_model.trainable = False

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
print("\n\n\n")
print("Summary of Final Model for Transfer Learning")
print("\n\n\n")
model.summary()

history = model.fit(data_tr,
                    epochs=3,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

'''
Fine-Tuning
'''

# Freeze the convolutional base
base_model.trainable = True

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
print("\n\n\n")
print("Summary of Final Model for Fine Tuning")
print("\n\n\n")
model.summary()

history = model.fit(data_tr,
                    epochs=2,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

# Evaluate the model
loss_vl, accuracy_vl = model.evaluate(data_vl)
print("\n\n\n")
print(f'Test Loss: {loss_vl}, Test Accuracy: {accuracy_vl}')
print("\n\n\n")

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()


### **2.3. ResNets**


> [Residual Networks](https://arxiv.org/abs/1512.03385) (ResNets) are a type of NN architecture that were introduced to address the problem of training very deep NNs. Prior to ResNets, as NNs became deeper, they became increasingly difficult to train due to issues like vanishing & exploding gradients. ResNets tackle this problem through the introduction of "skip connections" or "shortcut connections." These connections allow the network to skip one or more layers, offering an alternative path for the gradient during backpropagation. This design significantly mitigates the vanishing gradient problem, as it ensures that the gradient can be propagated back through the network more effectively. As a result, ResNets can be built with a much greater depth than was previously feasible, with models having layers in the hundreds, or even thousands, becoming trainable.

> [ResNets](https://www.kaggle.com/datasets/keras/resnet50) have been widely adopted in various fields of computer vision, such as image classification, object detection, & segmentation, due to their effectiveness & efficiency. The main advantage of ResNets is their ability to enable the training of very deep networks, which can lead to improved performance in complex tasks without suffering from the training difficulties associated with increased depth. The use of skip connections also helps in alleviating the problem of overfitting to some extent, as it simplifies the learned models. This makes ResNets particularly powerful for tasks that benefit from deep feature extraction, & they often achieve state-of-the-art results in many benchmarks & competitions.

> However, ResNets also have some limitations. While the skip connections solve the vanishing gradient problem, they can lead to an increase in computational complexity & memory usage, making them resource-intensive, especially as the network depth increases. This can be a drawback in scenarios where computational resources are limited or where real-time performance is necessary. Additionally, while ResNets have significantly improved the training of deep networks, they do not inherently solve other challenges in NN training, such as the need for large amounts of labeled data or the potential for biased training due to unrepresentative datasets. Despite these challenges, ResNets remain one of the most influential architectures in deep learning, particularly in the field of computer vision.

In [ ]:
#@title Example of ResNet50
'''
Runtime: GPU $$$
TensorFlow Standard Models:             https://keras.io/api/applications/
tf.keras.applications.ResNet50:         https://www.tensorflow.org/api_docs/python/tf/keras/applications/ResNet50
tf.keras.datasets.cifar10:              https://www.tensorflow.org/api_docs/python/tf/keras/datasets/cifar10
tf.keras.utils.to_categorical:          https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical
tf.data.Dataset.from_tensor_slices:     https://www.tensorflow.org/api_docs/python/tf/data/Dataset#from_tensor_slices
tf.data.Dataset.shuffle:                https://www.tensorflow.org/api_docs/python/tf/data/Dataset#shuffle
tf.data.Dataset.batch:                  https://www.tensorflow.org/api_docs/python/tf/data/Dataset#batch
tf.data.Dataset.prefetch:               https://www.tensorflow.org/api_docs/python/tf/data/Dataset#prefetch
tf.keras.Input:                         https://www.tensorflow.org/api_docs/python/tf/keras/Input
tf.keras.layers.GlobalAveragePooling2D: https://www.tensorflow.org/api_docs/python/tf/keras/layers/GlobalAveragePooling2D
tf.keras.layers.Dense:                  https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.layers.Dropout:                https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout
tf.keras.Model:                         https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.keras.callbacks.EarlyStopping:       https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
tf.keras.callbacks.ReduceLROnPlateau:   https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau
'''
import tensorflow as tf

# Load & preprocess CIFAR-10 dataset
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.cifar10.load_data()
datain_tr, datain_vl = datain_tr / 255.0, datain_vl / 255.0
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

def fun_create_dataset(data, labels, batch_size=32, train=True):
    dataset = tf.data.Dataset.from_tensor_slices((data, labels))
    if train: dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size)
    return dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

# Create training & validation datasets
data_tr = fun_create_dataset(datain_tr, dataou_tr, batch_size=32, train=True)
data_vl = fun_create_dataset(datain_vl, dataou_vl, batch_size=32, train=False)

# Load the base model with ImageNet weights
base_model = tf.keras.applications.ResNet50(include_top=False,
                                            weights='imagenet',
                                            input_shape=(32, 32, 3))

# Base model summary
print("\n\n\n")
print("Summary of Base Model")
print("\n\n\n")
base_model.summary()

# Create the model
inputs  = tf.keras.Input(shape=(32, 32, 3))
x       = base_model(inputs)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dense(1024, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
model   = tf.keras.Model(inputs=inputs, outputs=outputs, name='model')

# Set up callbacks for early stopping & learning rate reduction
cb_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                     patience=3,
                                                     restore_best_weights=True)

cb_reduce_lr      = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
                                                         factor=0.5,
                                                         patience=2,
                                                         min_lr=0.00001)

'''
Transfer Learning
'''

# Freeze the convolutional base
base_model.trainable = False

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
print("\n\n\n")
print("Summary of Final Model for Transfer Learning")
print("\n\n\n")
model.summary()

history = model.fit(data_tr,
                    epochs=3,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

'''
Fine-Tuning
'''

# Freeze the convolutional base
base_model.trainable = True

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.00001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
print("\n\n\n")
print("Summary of Final Model for Fine Tuning")
print("\n\n\n")
model.summary()

history = model.fit(data_tr,
                    epochs=2,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

# Evaluate the model
loss_vl, accuracy_vl = model.evaluate(data_vl)
print("\n\n\n")
print(f'Test Loss: {loss_vl}, Test Accuracy: {accuracy_vl}')
print("\n\n\n")

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()


### **2.4. DenseNets**

> [Dense Convolutional Networks](https://arxiv.org/abs/1608.06993) (DenseNets) introduce the idea of feature reuse, which is achieved through direct connections from any layer to all subsequent layers. This dense connectivity pattern ensures that each layer receives the feature maps of all preceding layers as input, fostering deeper feature propagation & encouraging feature reuse throughout the network. This [architectural design]([DenseNets](https://www.kaggle.com/code/abhranta/brain-tumor-detection-densenet-121)) was proposed to address the vanishing gradient problem, facilitate feature propagation, & reduce the number of parameters in the network, as each layer can leverage the feature maps of all previous layers.

> One of the major advantages of DenseNets is their efficiency in terms of parameters & computation. Due to the dense connections, each layer in a DenseNet has access to all preceding layers' feature maps, which means that the network can be more compact & require fewer parameters compared to traditional architectures with the same depth. This compactness also leads to a reduction in overfitting, making DenseNets particularly effective for tasks with limited training data. Additionally, the feature reuse mechanism enhances feature propagation & learning, allowing DenseNets to achieve high performance on various tasks, particularly in image classification & recognition.

> However, DenseNets come with certain drawbacks. The dense connectivity leads to a substantial increase in memory consumption during training, as the feature maps of all layers need to be stored for backpropagation. This can make training DenseNets on large datasets or deep architectures resource-intensive, requiring significant memory, which might be a limitation for systems with constrained computational resources. Furthermore, while the dense connections improve feature propagation, they also make the network architecture more complex, which can be a challenge for implementation & optimization. Despite these challenges, DenseNets represent a significant step forward in CNN architecture design, offering an efficient & effective approach for deep learning tasks in computer vision & beyond.

In [ ]:
#@title Example of DenseNet121
'''
Runtime: GPU $$$
TensorFlow Standard Models:             https://keras.io/api/applications/
tf.keras.applications.DenseNet121:      https://www.tensorflow.org/api_docs/python/tf/keras/applications/DenseNet121
tf.keras.datasets.cifar10:              https://www.tensorflow.org/api_docs/python/tf/keras/datasets/cifar10
tf.keras.utils.to_categorical:          https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical
tf.data.Dataset.from_tensor_slices:     https://www.tensorflow.org/api_docs/python/tf/data/Dataset#from_tensor_slices
tf.data.Dataset.shuffle:                https://www.tensorflow.org/api_docs/python/tf/data/Dataset#shuffle
tf.data.Dataset.batch:                  https://www.tensorflow.org/api_docs/python/tf/data/Dataset#batch
tf.data.Dataset.prefetch:               https://www.tensorflow.org/api_docs/python/tf/data/Dataset#prefetch
tf.keras.Input:                         https://www.tensorflow.org/api_docs/python/tf/keras/Input
tf.keras.layers.GlobalAveragePooling2D: https://www.tensorflow.org/api_docs/python/tf/keras/layers/GlobalAveragePooling2D
tf.keras.layers.Dense:                  https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.layers.Dropout:                https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout
tf.keras.Model:                         https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.keras.callbacks.EarlyStopping:       https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
tf.keras.callbacks.ReduceLROnPlateau:   https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau
'''
import tensorflow as tf

# Load & preprocess CIFAR-10 dataset
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.cifar10.load_data()
datain_tr, datain_vl = datain_tr / 255.0, datain_vl / 255.0
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

def fun_create_dataset(data, labels, batch_size=32, train=True):
    dataset = tf.data.Dataset.from_tensor_slices((data, labels))
    if train: dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size)
    return dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

# Create training & validation datasets
data_tr = fun_create_dataset(datain_tr, dataou_tr, batch_size=64, train=True)
data_vl = fun_create_dataset(datain_vl, dataou_vl, batch_size=64, train=False)

# Load the base model with ImageNet weights
base_model = tf.keras.applications.DenseNet121(include_top=False,
                                               weights='imagenet',
                                               input_shape=(32, 32, 3))

# Base model summary
print("\n\n\n")
print("Summary of Base Model")
print("\n\n\n")
base_model.summary()

# Create the model
inputs  = tf.keras.Input(shape=(32, 32, 3))
x       = base_model(inputs)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dense(512, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
model   = tf.keras.Model(inputs=inputs, outputs=outputs, name='model')

# Set up callbacks for early stopping & learning rate reduction
cb_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                     patience=3,
                                                     restore_best_weights=True)

cb_reduce_lr      = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
                                                         factor=0.5,
                                                         patience=2,
                                                         min_lr=0.00001)

'''
Transfer Learning
'''

# Freeze the convolutional base
base_model.trainable = False

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
print("\n\n\n")
print("Summary of Final Model for Transfer Learning")
print("\n\n\n")
model.summary()

history = model.fit(data_tr,
                    epochs=3,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

'''
Fine-Tuning
'''

# Freeze the convolutional base
base_model.trainable = True

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
print("\n\n\n")
print("Summary of Final Model for Fine Tuning")
print("\n\n\n")
model.summary()

history = model.fit(data_tr,
                    epochs=2,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

# Evaluate the model
loss_vl, accuracy_vl = model.evaluate(data_vl)
print("\n\n\n")
print(f'Test Loss: {loss_vl}, Test Accuracy: {accuracy_vl}')
print("\n\n\n")

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()


### **2.5. MobileNets**

> [MobileNets](https://arxiv.org/abs/1704.04861) are a class of efficient convolutional NNs (CNNs) designed specifically for mobile & edge devices, where computational resources are limited. Developed by Google researchers, the primary goal of MobileNets is to provide a balance between performance & efficiency, enabling the deployment of high-quality deep learning models on devices with less computational power & memory, such as smartphones, tablets, & IoT devices. To achieve this, MobileNets utilize streamlined architectures that use depthwise separable convolutions. This type of convolution splits the standard convolutional process into two layers: a depthwise convolution & a pointwise convolution, significantly reducing the computational cost & the number of parameters compared to standard convolutions found in more complex models.

> The efficiency of MobileNets makes them particularly well-suited for real-time applications & on-device ML tasks. Their reduced size & computational requirements allow for quicker inference times & lower power consumption, which are critical in mobile environments. Despite the reduction in complexity, MobileNets still manage to deliver competitive performance on a variety of tasks, including image classification, object detection, & face recognition. This balance of efficiency & performance opens up new possibilities for advanced AI applications on mobile devices, such as real-time image recognition & augmented reality experiences.

> However, the efficiency of MobileNets comes with trade-offs. The reduction in size & computational complexity can lead to lower accuracy compared to more complex & computationally intensive models like ResNets or Inception networks. The depthwise separable convolutions, while efficient, may not capture as much information as standard convolutions, potentially leading to a loss in model effectiveness for more complex tasks or datasets. Additionally, the design & optimization of MobileNets require careful consideration of the balance between accuracy & efficiency, which can vary depending on the specific application & device capabilities. Despite these limitations, MobileNets remain a popular choice for on-device ML applications, offering a practical solution for deploying deep learning models in resource-constrained environments.

In [ ]:
#@title Example of MobileNets
'''
Runtime: GPU $$$
TensorFlow Standard Models:             https://keras.io/api/applications/
tf.keras.applications.MobileNet:      https://www.tensorflow.org/api_docs/python/tf/keras/applications/MobileNet
tf.keras.datasets.cifar10:              https://www.tensorflow.org/api_docs/python/tf/keras/datasets/cifar10
tf.keras.utils.to_categorical:          https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical
tf.data.Dataset.from_tensor_slices:     https://www.tensorflow.org/api_docs/python/tf/data/Dataset#from_tensor_slices
tf.data.Dataset.shuffle:                https://www.tensorflow.org/api_docs/python/tf/data/Dataset#shuffle
tf.data.Dataset.batch:                  https://www.tensorflow.org/api_docs/python/tf/data/Dataset#batch
tf.data.Dataset.prefetch:               https://www.tensorflow.org/api_docs/python/tf/data/Dataset#prefetch
tf.keras.Input:                         https://www.tensorflow.org/api_docs/python/tf/keras/Input
tf.keras.layers.GlobalAveragePooling2D: https://www.tensorflow.org/api_docs/python/tf/keras/layers/GlobalAveragePooling2D
tf.keras.layers.Dense:                  https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.layers.Dropout:                https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout
tf.keras.Model:                         https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.keras.callbacks.EarlyStopping:       https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
tf.keras.callbacks.ReduceLROnPlateau:   https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau
'''
import tensorflow as tf

# Load & preprocess CIFAR-10 dataset
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.cifar10.load_data()
datain_tr, datain_vl = datain_tr / 255.0, datain_vl / 255.0
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

def fun_create_dataset(data, labels, batch_size=32, train=True):
    dataset = tf.data.Dataset.from_tensor_slices((data, labels))
    if train: dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size)
    return dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

# Create training & validation datasets
data_tr = fun_create_dataset(datain_tr, dataou_tr, batch_size=64, train=True)
data_vl = fun_create_dataset(datain_vl, dataou_vl, batch_size=64, train=False)

# Load the base model with ImageNet weights
base_model = tf.keras.applications.DenseNet121(include_top=False,
                                               weights='imagenet',
                                               input_shape=(32, 32, 3))

# Base model summary
print("\n\n\n")
print("Summary of Base Model")
print("\n\n\n")
base_model.summary()

# Create the model
inputs  = tf.keras.Input(shape=(32, 32, 3))
x       = base_model(inputs)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dense(512, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
model   = tf.keras.Model(inputs=inputs, outputs=outputs, name='model')

# Set up callbacks for early stopping & learning rate reduction
cb_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                     patience=3,
                                                     restore_best_weights=True)

cb_reduce_lr      = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
                                                         factor=0.5,
                                                         patience=2,
                                                         min_lr=0.00001)

'''
Transfer Learning
'''

# Freeze the convolutional base
base_model.trainable = False

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
print("\n\n\n")
print("Summary of Final Model for Transfer Learning")
print("\n\n\n")
model.summary()

history = model.fit(data_tr,
                    epochs=3,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

'''
Fine-Tuning
'''

# Freeze the convolutional base
base_model.trainable = True

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
print("\n\n\n")
print("Summary of Final Model for Fine Tuning")
print("\n\n\n")
model.summary()

history = model.fit(data_tr,
                    epochs=2,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

# Evaluate the model
loss_vl, accuracy_vl = model.evaluate(data_vl)
print("\n\n\n")
print(f'Test Loss: {loss_vl}, Test Accuracy: {accuracy_vl}')
print("\n\n\n")

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()


### **2.6. EfficientNets**


> [EfficientNets](https://arxiv.org/abs/1905.11946) are a series of convolutional NN (CNN) architectures designed to provide an optimized balance between accuracy & efficiency in deep learning models. Introduced by researchers at Google, the primary goal of EfficientNets is to achieve higher accuracy while being more computationally efficient than existing CNN architectures. This is particularly important for deploying deep learning models in environments where computational resources are limited, such as mobile devices or edge computing platforms. EfficientNets achieve this balance through a systematic study of scaling network width, depth, & resolution, which led to the development of a set of scaling rules. These rules guide how to proportionally increase the network's depth, width, & input image resolution, ensuring that the increase in complexity & computational cost is balanced with the gains in model performance.

> EfficientNets are built upon the baseline architecture called EfficientNet-B0, which was initially optimized for accuracy & efficiency using neural architecture search. Subsequent versions, from B1 to B7, apply the scaling rules to this baseline model, gradually increasing the depth, width, & resolution. This results in a family of models that offer a range of trade-offs between accuracy & efficiency. EfficientNets have shown remarkable performance on benchmarks for tasks like image classification, often outperforming larger & more complex models. The efficiency of these models makes them suitable for a variety of applications, especially where computational resources are a bottleneck.

> However, there are some drawbacks to the EfficientNet architecture. While the models are designed to be more efficient than other architectures of similar accuracy, they can still be computationally intensive, especially the larger variants like B5 to B7. Training these models requires significant computational resources, which might not be feasible for individuals or organizations with limited hardware capabilities. Moreover, the process of scaling up the network dimensions while maintaining efficiency & effectiveness requires careful tuning & might not always yield proportional improvements in performance across different tasks or datasets. Despite these challenges, EfficientNets represent a significant advancement in the development of efficient & high-performing CNN architectures, offering a promising solution for deploying powerful deep learning models in resource-constrained environments

In [ ]:
#@title Example of EfficientNets
'''
Runtime: GPU $$$
TensorFlow Standard Models:             https://keras.io/api/applications/
tf.keras.applications.EfficientNetB0:   https://www.tensorflow.org/api_docs/python/tf/keras/applications/EfficientNetB0
tf.keras.datasets.cifar10:              https://www.tensorflow.org/api_docs/python/tf/keras/datasets/cifar10
tf.keras.utils.to_categorical:          https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical
tf.data.Dataset.from_tensor_slices:     https://www.tensorflow.org/api_docs/python/tf/data/Dataset#from_tensor_slices
tf.data.Dataset.shuffle:                https://www.tensorflow.org/api_docs/python/tf/data/Dataset#shuffle
tf.data.Dataset.batch:                  https://www.tensorflow.org/api_docs/python/tf/data/Dataset#batch
tf.data.Dataset.prefetch:               https://www.tensorflow.org/api_docs/python/tf/data/Dataset#prefetch
tf.keras.Input:                         https://www.tensorflow.org/api_docs/python/tf/keras/Input
tf.keras.layers.GlobalAveragePooling2D: https://www.tensorflow.org/api_docs/python/tf/keras/layers/GlobalAveragePooling2D
tf.keras.layers.Dense:                  https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.layers.Dropout:                https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout
tf.keras.Model:                         https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.keras.callbacks.EarlyStopping:       https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
tf.keras.callbacks.ReduceLROnPlateau:   https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau
'''
import tensorflow as tf

# Load & preprocess CIFAR-10 dataset
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.cifar10.load_data()
datain_tr, datain_vl = datain_tr / 255.0, datain_vl / 255.0
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

def fun_create_dataset(data, labels, batch_size=32, train=True):
    dataset = tf.data.Dataset.from_tensor_slices((data, labels))
    if train: dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size)
    return dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

# Create training & validation datasets
data_tr = fun_create_dataset(datain_tr, dataou_tr, batch_size=64, train=True)
data_vl = fun_create_dataset(datain_vl, dataou_vl, batch_size=64, train=False)

# Load the base model with ImageNet weights
base_model = tf.keras.applications.DenseNet121(include_top=False,
                                               weights='imagenet',
                                               input_shape=(32, 32, 3))

# Base model summary
print("\n\n\n")
print("Summary of Base Model")
print("\n\n\n")
base_model.summary()

# Create the model
inputs  = tf.keras.Input(shape=(32, 32, 3))
x       = base_model(inputs)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dense(512, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
model   = tf.keras.Model(inputs=inputs, outputs=outputs, name='model')

# Set up callbacks for early stopping & learning rate reduction
cb_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                     patience=3,
                                                     restore_best_weights=True)

cb_reduce_lr      = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
                                                         factor=0.5,
                                                         patience=2,
                                                         min_lr=0.00001)

'''
Transfer Learning
'''

# Freeze the convolutional base
base_model.trainable = False

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
print("\n\n\n")
print("Summary of Final Model for Transfer Learning")
print("\n\n\n")
model.summary()

history = model.fit(data_tr,
                    epochs=3,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

'''
Fine-Tuning
'''

# Freeze the convolutional base
base_model.trainable = True

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
print("\n\n\n")
print("Summary of Final Model for Fine Tuning")
print("\n\n\n")
model.summary()

history = model.fit(data_tr,
                    epochs=2,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

# Evaluate the model
loss_vl, accuracy_vl = model.evaluate(data_vl)
print("\n\n\n")
print(f'Test Loss: {loss_vl}, Test Accuracy: {accuracy_vl}')
print("\n\n\n")

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()


### **2.7. Xception**

> [Xception](https://arxiv.org/abs/1610.02357) which stands for "Extreme Inception," is a deep learning NN architecture that modifies & extends the Inception architecture. It was introduced by François Chollet, the creator of the Keras deep learning library. The primary innovation of Xception is the introduction of depthwise separable convolutions, which is a more extreme version of the Inception model's idea of factorizing convolutions into smaller, more manageable operations. Depthwise separable convolutions separate the learning of spatial features (depthwise convolution) from the learning of channel-wise features (pointwise convolution), leading to a more efficient use of model parameters. This architecture is designed to improve upon the performance of Inception networks in terms of accuracy & computational efficiency, particularly in tasks involving large-scale image recognition.

> Xception's structure allows it to have a lower number of parameters compared to conventional convolutional NNs of similar depth, making it more efficient in terms of computational resources. This efficiency makes Xception an attractive model for scenarios where computational resources are limited, such as mobile or edge computing. Despite its reduced parameter count, Xception demonstrates competitive or even superior performance in image classification tasks compared to other advanced architectures, owing to its efficient processing of spatial & channel-wise features.

> However, Xception also has some limitations. While it is more efficient than traditional CNNs, it still requires significant computational power to train, which can be a barrier for those without access to high-performance computing resources. Additionally, the architecture's focus on depthwise separable convolutions may not always be the best approach for all types of data or tasks; in some cases, traditional convolutions might yield better results. The model's unique architecture also requires a deeper understanding of its components for effective customization & optimization, potentially making it less accessible for beginners in deep learning. Despite these challenges, Xception stands as a powerful & efficient architecture in the landscape of convolutional NNs, especially for large-scale image processing tasks.

In [ ]:
#@title Example of Xception
'''
Runtime: GPU $$$
TensorFlow Standard Models:             https://keras.io/api/applications/
tf.keras.applications.Xception:         https://www.tensorflow.org/api_docs/python/tf/keras/applications/Xception
tf.keras.datasets.cifar10:              https://www.tensorflow.org/api_docs/python/tf/keras/datasets/cifar10
tf.keras.utils.to_categorical:          https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical
tf.data.Dataset.from_tensor_slices:     https://www.tensorflow.org/api_docs/python/tf/data/Dataset#from_tensor_slices
tf.data.Dataset.shuffle:                https://www.tensorflow.org/api_docs/python/tf/data/Dataset#shuffle
tf.data.Dataset.batch:                  https://www.tensorflow.org/api_docs/python/tf/data/Dataset#batch
tf.data.Dataset.prefetch:               https://www.tensorflow.org/api_docs/python/tf/data/Dataset#prefetch
tf.keras.Input:                         https://www.tensorflow.org/api_docs/python/tf/keras/Input
tf.keras.layers.GlobalAveragePooling2D: https://www.tensorflow.org/api_docs/python/tf/keras/layers/GlobalAveragePooling2D
tf.keras.layers.Dense:                  https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.layers.Dropout:                https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout
tf.keras.Model:                         https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.keras.callbacks.EarlyStopping:       https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
tf.keras.callbacks.ReduceLROnPlateau:   https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau
'''
import tensorflow as tf

# Load & preprocess CIFAR-10 dataset
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.cifar10.load_data()
datain_tr, datain_vl = datain_tr / 255.0, datain_vl / 255.0
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

def fun_create_dataset(data, labels, batch_size=32, train=True):
    dataset = tf.data.Dataset.from_tensor_slices((data, labels))
    if train: dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size)
    return dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

# Create training & validation datasets
data_tr = fun_create_dataset(datain_tr, dataou_tr, batch_size=64, train=True)
data_vl = fun_create_dataset(datain_vl, dataou_vl, batch_size=64, train=False)

# Load the base model with ImageNet weights
base_model = tf.keras.applications.Xception(include_top=False,
                                            weights='imagenet',
                                            input_shape=(71, 71, 3))

# Base model summary
print("\n\n\n")
print("Summary of Base Model")
print("\n\n\n")
base_model.summary()

# Create the model
inputs  = tf.keras.Input(shape=(32, 32, 3))
x       = tf.keras.layers.Resizing(71,71)(inputs)
x       = base_model(x)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dense(512, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
model   = tf.keras.Model(inputs=inputs, outputs=outputs, name='model')

# Set up callbacks for early stopping & learning rate reduction
cb_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                     patience=3,
                                                     restore_best_weights=True)

cb_reduce_lr      = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
                                                         factor=0.5,
                                                         patience=2,
                                                         min_lr=0.00001)

'''
Transfer Learning
'''

# Freeze the convolutional base
base_model.trainable = False

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
print("\n\n\n")
print("Summary of Final Model for Transfer Learning")
print("\n\n\n")
model.summary()

history = model.fit(data_tr,
                    epochs=3,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

'''
Fine-Tuning
'''

# Freeze the convolutional base
base_model.trainable = True

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
print("\n\n\n")
print("Summary of Final Model for Fine Tuning")
print("\n\n\n")
model.summary()

history = model.fit(data_tr,
                    epochs=2,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

# Evaluate the model
loss_vl, accuracy_vl = model.evaluate(data_vl)
print("\n\n\n")
print(f'Test Loss: {loss_vl}, Test Accuracy: {accuracy_vl}')
print("\n\n\n")

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()


### **2.8. NASNets**

> [NASNet](https://arxiv.org/abs/1707.07012) (Neural Architecture Search Network) is a convolutional NN architecture that was developed through a process known as neural architecture search (NAS). This process involves using ML algorithms to automatically design new NN architectures, with the goal of finding models that are both highly efficient & performant. NASNet was specifically developed to optimize for accuracy & scalability across a variety of image recognition tasks. The use of NAS allows for the automated discovery of architectural building blocks that are well-suited for specific tasks, which can then be scaled to different computational budgets. This approach contrasts with traditional methods where network architectures are manually designed & tweaked by human experts.

> The architecture of NASNet is characterized by its modularity, with repeatable cells that are optimized for both normal image processing & reduction of spatial resolution. These cells are discovered through the NAS process, where reinforcement learning is used to explore a large space of potential architectures. By optimizing these cells for different tasks & computational constraints, NASNet can be adapted for various applications, from mobile devices with limited computational capabilities to larger systems that require more processing power. This flexibility is one of the main advantages of NASNet, allowing for a single architecture to be efficiently scaled across a wide range of use cases.

> However, NASNet & the NAS process, in general, come with certain drawbacks. The process of searching for optimal architectures is computationally intensive & time-consuming, requiring significant resources. This can make NAS impractical for smaller organizations or individuals without access to large-scale computing infrastructure. Additionally, while NAS can automate the discovery of effective architectures, it may not always produce models that are easily interpretable or modifiable by humans, potentially limiting the ability to manually fine-tune or understand the specific design choices of the network. Despite these challenges, NASNet represents a significant step forward in the automated design of NNs, demonstrating the potential of leveraging ML not just for performing tasks, but also for designing the architectures that perform these tasks.

## **3. Memory-Efficient Models**

When designing artificial intelligence systems, a critical consideration is creating models that use memory effectively. Such models are especially important in devices with limited storage capacities. A memory-efficient model strikes a balance between complexity and size, ensuring that it can perform the required tasks without consuming excessive system resources.

### **3.1 SqueezeNet**



> [SqueezeNet](https://arxiv.org/abs/1602.07360) is a type of convolutional NN (CNN) architecture that was developed with a focus on reducing the model size while maintaining competitive performance, particularly in image classification tasks. The primary objective of SqueezeNet is to achieve AlexNet-level accuracy on the ImageNet dataset with a fraction of the model parameters. This reduction in size is crucial for deploying deep learning models in environments with stringent constraints on memory & computational power, such as mobile or embedded devices. SqueezeNet achieves this through an architectural design that emphasizes the use of 1x1 convolution filters, an innovative 'squeeze' & 'expand' strategy in its layers, & a decrease in the number of input channels to 3x3 filters.

> The core of SqueezeNet's architecture is the 'Fire' module, which comprises a squeeze layer (using 1x1 convolutions) followed by an expand layer (combining 1x1 & 3x3 convolutions). The squeeze layer reduces the depth of the network, decreasing the number of parameters & computational requirements. The expand layer then increases the depth, allowing the network to capture more complex features from the input data. This design significantly reduces the number of parameters without a substantial loss in accuracy. SqueezeNet also employs strategies like delayed downsampling to maintain large activation maps, which help in retaining important spatial information for accurate classifications.

> However, SqueezeNet's aggressive reduction in parameters does come with trade-offs. While the model is highly efficient in terms of size, this efficiency can sometimes result in a drop in accuracy, especially when compared to larger & more complex models. The focus on 1x1 convolutions, though beneficial for reducing parameters, might limit the model's ability to capture more complex patterns & relationships in the data, which larger filters might be able to detect. Additionally, the unique architecture of SqueezeNet, while innovative, may not be as versatile or adaptable for tasks outside of image classification. Despite these limitations, SqueezeNet has been influential in demonstrating that it is possible to significantly reduce the size of a NN without a proportional decrease in performance, a crucial consideration for deep learning applications in resource-constrained environments.

In [ ]:
#@title Example of SqueezeNet
'''
Runtime: GPU $$$
TensorFlow Standard Models: https://keras.io/api/applications/
Tensorflow Callbacks:        https://www.tensorflow.org/api_docs/python/tf/keras/callbacks
'''

import tensorflow as tf

# Load & preprocess CIFAR-10 dataset
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.cifar10.load_data()
datain_tr, datain_vl = datain_tr / 255.0, datain_vl / 255.0
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

def fun_create_dataset(data, labels, batch_size=32, train=True):
    dataset = tf.data.Dataset.from_tensor_slices((data, labels))
    if train: dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size)
    return dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

# Create training & validation datasets
data_tr = fun_create_dataset(datain_tr, dataou_tr, batch_size=64, train=True)
data_vl = fun_create_dataset(datain_vl, dataou_vl, batch_size=64, train=False)

# SqueezeNet Fire Module
def fun_fire_module(x, squeeze_planes, expand_planes):
    squeeze = tf.keras.layers.Conv2D(squeeze_planes, (1, 1), padding='same', activation='relu')(x)
    expand_1x1 = tf.keras.layers.Conv2D(expand_planes, (1, 1), padding='same', activation='relu')(squeeze)
    expand_3x3 = tf.keras.layers.Conv2D(expand_planes, (3, 3), padding='same', activation='relu')(squeeze)
    return tf.keras.layers.concatenate([expand_1x1, expand_3x3], axis=3)

# SqueezeNet Model
def fun_squeezenet(input_shape):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Conv2D(96, (7, 7), strides=(2, 2), padding='same', activation='relu')(inputs)
    x = tf.keras.layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2))(x)
    x = fun_fire_module(x, 16, 64)
    x = fun_fire_module(x, 16, 64)
    x = fun_fire_module(x, 32, 128)
    x = tf.keras.layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2))(x)
    x = fun_fire_module(x, 32, 128)
    x = fun_fire_module(x, 48, 192)
    x = fun_fire_module(x, 48, 192)
    x = fun_fire_module(x, 64, 256)
    x = tf.keras.layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2))(x)
    x = fun_fire_module(x, 64, 256)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Conv2D(10, (1, 1), padding='same', activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    outputs = tf.keras.layers.Activation('softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    return model

# Create the model
model = fun_squeezenet(input_shape=(32, 32, 3))

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Model summary
model.summary()

# Set up callbacks for early stopping & learning rate reduction
cb_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                     patience=3,
                                                     restore_best_weights=True)

cb_reduce_lr      = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
                                                         factor=0.5,
                                                         patience=2,
                                                         min_lr=0.00001)

history = model.fit(data_tr,
                    epochs=10,
                    validation_data=data_vl,
                    callbacks=[cb_early_stopping, cb_reduce_lr])

# Evaluate the model
loss_vl, accuracy_vl = model.evaluate(data_vl)
print(f'Test Loss: {loss_vl}, Test Accuracy: {accuracy_vl}')

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **3.2 Capsule Networks**


> Introduced by Geoffrey Hinton & his team, [Capsule Networks](https://proceedings.neurips.cc/paper_files/paper/2017/file/2cad8fa47bbef282badbb8de5374b894-Paper.pdf) (CapsNets) aim to add more structure & hierarchy to the way NNs process visual information. Traditional CNNs, while effective in many scenarios, often struggle with understanding spatial hierarchies & relationships in an image, such as the relative positioning & orientation of objects. CapsNets tackle this issue by using capsules – groups of neurons that work together to detect objects & their parts. Each capsule is designed to recognize a specific type of image feature & to output a vector that represents the probability of the feature's presence as well as its properties (such as orientation & size). This vector representation allows the network to preserve detailed spatial information about the features.

> The key component of Capsule Networks is the dynamic routing algorithm, which replaces the pooling layers commonly found in CNNs. In a CNN, pooling layers are used to reduce the spatial dimensions of the data, which can lead to the loss of important spatial hierarchies & relationships. The dynamic routing algorithm in CapsNets, on the other hand, ensures that the capsules in one layer make predictions for capsules in the next layer, & these predictions are weighted based on their agreement. This process allows the network to understand & preserve the spatial relationships between different features in the image. As a result, Capsule Networks are particularly adept at understanding images with complex internal structures & can be more robust to variations in the viewpoint, rotation, & spatial arrangement of objects.

> Despite their innovative approach, Capsule Networks come with their own set of challenges. One significant issue is computational complexity. The dynamic routing process & the use of vector representations make CapsNets computationally more intensive than traditional CNNs, which can be a hindrance when working with large datasets or requiring real-time processing. Additionally, the implementation & optimization of CapsNets are more complex, & the research on these networks is still in a relatively early stage compared to the more mature field of CNNs. This means that best practices & effective techniques for training & deploying CapsNets are still being developed. Despite these challenges, Capsule Networks offer a promising direction for advancements in NN design, particularly in tasks that require a deep understanding of spatial hierarchies & relationships in visual data.

In [ ]:
#@title Example of Capsule Networks
'''
Runtime: GPU $$$
TensorFlow Standard Models: https://keras.io/api/applications/
Tensorflow Callbacks:       https://www.tensorflow.org/api_docs/python/tf/keras/callbacks
'''

import tensorflow as tf

# Load & prepare the MNIST dataset
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.mnist.load_data()
datain_tr = datain_tr.reshape(datain_tr.shape[0], 28, 28, 1).astype('float32') / 255
datain_vl = datain_vl.reshape(datain_vl.shape[0], 28, 28, 1).astype('float32') / 255
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

# Squash function
def squash(x, axis=-1):
    s_squared_norm = tf.reduce_sum(tf.square(x), axis, keepdims=True)
    scale = s_squared_norm / (1 + s_squared_norm) / tf.sqrt(s_squared_norm + tf.keras.backend.epsilon())
    return scale * x

# Capsule Layer
class CapsuleLayer(tf.keras.layers.Layer):
    def __init__(self, num_capsule, dim_capsule, routings=3, **kwargs):
        super(CapsuleLayer, self).__init__(**kwargs)
        self.num_capsule = num_capsule
        self.dim_capsule = dim_capsule
        self.routings = routings

    def build(self, input_shape):
        self.kernel = self.add_weight(shape=(input_shape[-1], self.num_capsule * self.dim_capsule),
                                      initializer='glorot_uniform',
                                      name='kernel',
                                      trainable=True)

    def call(self, inputs):
        inputs_expand = tf.expand_dims(inputs, 2)
        inputs_tiled = tf.tile(inputs_expand, [1, 1, self.num_capsule, 1])
        inputs_hat = tf.map_fn(lambda x: tf.matmul(x, self.kernel), elems=inputs_tiled)
        b = tf.zeros(shape=[tf.shape(inputs)[0], inputs.shape[1], self.num_capsule])

        for i in range(self.routings):
            c = tf.nn.softmax(b, axis=-1)
            s = tf.reduce_sum(c[..., tf.newaxis] * inputs_hat, axis=1)
            v = squash(s)
            if i < self.routings - 1:
                b += tf.reduce_sum(inputs_hat * v[:, tf.newaxis, :, :], axis=-1)
        return v

# Create the model
inputs = tf.keras.Input(shape=(28, 28, 1))
x = tf.keras.layers.Conv2D(256, (9, 9), strides=(1, 1), activation='relu', padding='valid')(inputs)
x = tf.keras.layers.Reshape([-1, 256])(x)
capsule = CapsuleLayer(10, 16, 3)(x)
output = tf.keras.layers.Lambda(lambda z: tf.sqrt(tf.reduce_sum(tf.square(z), 2)))(capsule)

model = tf.keras.Model(inputs=inputs, outputs=output)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=0.001)

# Train the model
history = model.fit(datain_tr, dataou_tr, batch_size=64, epochs=10, validation_data=(datain_vl, dataou_vl), callbacks=[early_stopping, reduce_lr])

# Evaluate the model
test_loss, test_acc = model.evaluate(datain_vl, dataou_vl, verbose=2)
print(f"Test accuracy: {test_acc}, Test loss: {test_loss}")

# Print the summary of the model
model.summary()

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

## **4. Segmentation & Object Detection Models**

Segmentation models divide images into parts or regions typically to identify objects or boundaries. For instance, an algorithm might delineate individual cells in a medical image or partition a street scene into cars, pedestrians, and buildings.

On the other hand, object detection models locate and identify objects within images. They draw bounding boxes around objects and label them. Imagine a box drawn around every person in a photograph, with each box labeled person.

> <div align="left">
  <img src="https://1.bp.blogspot.com/-HKhrGghm3Z4/Xwd6oWNmCnI/AAAAAAAADRQ/Hff-ZgjSDvo7op7aUtdN--WSuMohSMn-gCLcBGAsYHQ/s1600/tensorflow2objectdetection.png" width="50%">
  <br>
  <figcaption>Figure: Detection of Persons & Objects in a Given Image (<a href="https://1.bp.blogspot.com/-HKhrGghm3Z4/Xwd6oWNmCnI/AAAAAAAADRQ/Hff-ZgjSDvo7op7aUtdN--WSuMohSMn-gCLcBGAsYHQ/s1600/tensorflow2objectdetection.png">img</a>)</figcaption>
</div>

### **4.1. U-Net**

> [U-Net](https://arxiv.org/abs/1505.04597) is a type of convolutional NN (CNN) architecture that was specifically designed for biomedical image segmentation tasks. Developed by Olaf Ronneberger, Philipp Fischer, & Thomas Brox in 2015, U-Net was introduced to address the need for more precise & efficient segmentation in medical imaging, where the accurate delineation of relevant biological structures (like tumors or organs) is crucial. The architecture is characterized by its unique "U" shape, which is formed by a contracting path to capture context & a symmetric expanding path that enables precise localization. This design allows U-Net to effectively learn from a limited number of training samples, which is often a significant challenge in medical image analysis due to the scarcity of labeled data.

> The contracting (downsampling) path of U-Net consists of repeated application of convolutions & pooling operations, which helps the network in capturing the context in the images. The expanding (upsampling) path, on the other hand, combines the high-level features from the contracting path with the upsampled output, enabling precise localization & a high resolution of the output segmentation map. This is facilitated by skip connections that directly connect layers of the contracting path with corresponding layers in the expanding path. These skip connections are a crucial aspect of U-Net, as they allow the network to propagate context information to higher resolution layers, thereby combining semantic information from deep, coarse layers with appearance information from shallow, fine layers.

> Despite its effectiveness, U-Net does have some limitations. The model’s architecture, particularly the need for numerous skip connections, can lead to a significant consumption of memory & computational resources. This can be a challenge when dealing with very large images or when attempting to process images in real-time. Additionally, while U-Net has been highly successful in biomedical image segmentation, its specialized architecture may not be as directly applicable or optimal for other types of image analysis tasks or datasets that differ significantly from medical images. However, the fundamental principles of U-Net have inspired various adaptations & improvements, making it a highly influential architecture in the field of medical image analysis & beyond.

In [ ]:
#@title Example of U-Net
'''
Runtime: GPU $$$
tf.keras.layers.Conv2D:          https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D
tf.keras.layers.Activation:      https://www.tensorflow.org/api_docs/python/tf/keras/layers/Activation
tf.keras.layers.MaxPooling2D:    https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPooling2D
tf.keras.layers.Conv2DTranspose: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2DTranspose
tf.keras.layers.concatenate:     https://www.tensorflow.org/api_docs/python/tf/keras/layers/concatenate
tf.keras.layers.Input:           https://www.tensorflow.org/api_docs/python/tf/keras/layers/Input
tf.keras.models.Model:           https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.image.resize:                 https://www.tensorflow.org/api_docs/python/tf/image/resize
tf.cast:                         https://www.tensorflow.org/api_docs/python/tf/cast
tf.keras.models.Model.compile:   https://www.tensorflow.org/api_docs/python/tf/keras/Model#compile
tf.keras.models.Model.fit:        https://www.tensorflow.org/api_docs/python/tf/keras/Model#fit
tf.keras.models.Model.predict:   https://www.tensorflow.org/api_docs/python/tf/keras/Model#predict
tensorflow_datasets:              https://www.tensorflow.org/datasets
tfds.load:                       https://www.tensorflow.org/datasets/api_docs/python/tfds/load
'''
import tensorflow as tf
import tensorflow_datasets as tfds
from matplotlib import pyplot as plt

# Define a convolutional block function used in the U-Net architecture
def fun_conv_block(input_tensor, num_filters):
    # Apply a Convolutional Layer
    x = tf.keras.layers.Conv2D(num_filters, (3, 3), padding="same")(input_tensor)
    # Apply a ReLU activation function
    x = tf.keras.layers.Activation("relu")(x)
    # Apply another Convolutional Layer
    x = tf.keras.layers.Conv2D(num_filters, (3, 3), padding="same")(x)
    # Apply a ReLU activation function again
    x = tf.keras.layers.Activation("relu")(x)
    return x

# Define an encoder block function for the U-Net architecture
def fun_encoder_block(input_tensor, num_filters):
    # Use the convolutional block defined earlier
    x = fun_conv_block(input_tensor, num_filters)
    # Apply Max Pooling
    p = tf.keras.layers.MaxPooling2D((2, 2))(x)
    return x, p

# Define a decoder block function for the U-Net architecture
def fun_decoder_block(input_tensor, concat_tensor, num_filters):
    # Apply a transposed convolution (upscaling)
    x = tf.keras.layers.Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(input_tensor)
    # Concatenate with the corresponding encoder output
    x = tf.keras.layers.concatenate([x, concat_tensor])
    # Use the convolutional block again
    x = fun_conv_block(x, num_filters)
    return x

# Define the U-Net model
def fun_unet_model(input_size=(128, 128, 3)):
    # Define the input shape
    inputs = tf.keras.layers.Input(input_size)
    # Create the encoder blocks
    x1, p1 = fun_encoder_block(inputs, 64)
    x2, p2 = fun_encoder_block(p1, 128)
    x3, p3 = fun_encoder_block(p2, 256)
    x4, p4 = fun_encoder_block(p3, 512)
    # Apply convolution block at the bottom of the U-Net
    b = fun_conv_block(p4, 1024)
    # Create the decoder blocks
    d1 = fun_decoder_block(b, x4, 512)
    d2 = fun_decoder_block(d1, x3, 256)
    d3 = fun_decoder_block(d2, x2, 128)
    d4 = fun_decoder_block(d3, x1, 64)
    # Define the output layer
    outputs = tf.keras.layers.Conv2D(3, (1, 1), padding="same", activation="softmax")(d4)
    # Create the model
    model = tf.keras.models.Model(inputs, outputs)
    return model

# Load & preprocess the Oxford-IIIT Pet Dataset
def fun_load_pet_dataset(batch_size=32):
    # Load the dataset using TensorFlow Datasets
    dataset, info = tfds.load('oxford_iiit_pet:3.*.*', with_info=True)

    # Function to preprocess the dataset
    def fun_preprocess(item):
        # Resize the image & mask
        image = tf.image.resize(item['image'], (128, 128))
        mask = tf.image.resize(item['segmentation_mask'], (128, 128))
        # Adjust mask values & cast to float
        mask = tf.cast(mask, tf.float32)
        mask = mask - 1
        return image, mask

    # Apply preprocessing, batching, & prefetching
    data_tr = dataset['train'].map(fun_preprocess).batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)
    data_vl = dataset['test'].map(fun_preprocess).batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)

    return data_tr, data_vl

# Load training & validation data
data_tr, data_vl = fun_load_pet_dataset(batch_size=32)

# Create & compile the U-Net model
model = fun_unet_model(input_size=(128, 128, 3))
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Print the model summary
model.summary()

# Train the model
model.fit(data_tr, epochs=5, validation_data=data_vl)

# Function to display segmentation examples
def fun_segmentation_examples(data_vl, data_es):
    # Take one batch of validation data
    for images, masks in data_vl.take(1):
        # Select first 5 images & masks
        images, masks, estimates = images[:5], masks[:5], data_es[:5,...]
        # Create subplots
        fig, ax = plt.subplots(3, 5, figsize=(15, 9))
        for i0, (image, mask) in enumerate(zip(images, masks)):
            # Calculate the estimated mask
            estimate = tf.expand_dims(estimates[i0,...].max(axis=-1), axis=-1)
            # Show the original image, true mask, & estimated mask
            ax[0,i0].imshow(image / tf.reduce_max(image))
            ax[1,i0].imshow(mask / tf.reduce_max(mask), cmap='gray')
            ax[2,i0].imshow(estimate / tf.reduce_max(estimate), cmap='gray')

            # Remove axis ticks
            ax[0,i0].set_xticks([])
            ax[0,i0].set_yticks([])
            ax[1,i0].set_xticks([])
            ax[1,i0].set_yticks([])
            ax[2,i0].set_xticks([])
            ax[2,i0].set_yticks([])
        # Adjust subplot parameters
        plt.subplots_adjust(wspace=0, hspace=0)
        plt.show()

# Generate predictions for the validation data
data_es = model.predict(data_vl)

# Display segmentation examples
fun_segmentation_examples(data_vl, data_es)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **4.2. Yolo (You Only Look Once)**


> [YOLO](https://arxiv.org/abs/1506.02640) (You Only Look Once) YOLO (You Only Look Once) is an object detection system that represents a significant shift in approach from traditional two-step object detection methods. Traditional methods typically involve a region proposal step followed by a classification step. YOLO, on the other hand, unifies these steps into a single end-to-end model. It divides the input image into a grid, & for each grid cell, it simultaneously predicts multiple bounding boxes & class probabilities for those boxes. This approach allows YOLO to predict the presence & locations of multiple objects in a single forward pass through the network, dramatically increasing the speed of detection.

> One of the primary advantages of YOLO is its speed, making it highly suitable for real-time applications. Since it processes the entire image at once & requires only a single forward pass through the network to make predictions, YOLO can achieve much higher frame rates compared to region-proposal-based methods. This makes it an ideal choice for applications where real-time detection is crucial, such as in video surveillance, autonomous vehicles, & other areas where quick, accurate detection is essential. Furthermore, because YOLO processes the entire image globally, it tends to make fewer false positives on background areas, as it learns contextual information about object classes & their appearance within the broader image context.

> However, YOLO also has some drawbacks. One of the main limitations is its relative weakness in detecting small objects within an image, as well as objects that appear in groups or are closely clustered together. This is partly due to its spatial constraints imposed by the grid-based approach. Additionally, the early versions of YOLO tended to be less accurate in terms of bounding box localization compared to more complex two-step methods. This has been addressed in later versions & variants of YOLO, but it remains a consideration when choosing an object detection model. Despite these challenges, YOLO's speed & efficiency have made it a popular choice in the field of computer vision, particularly in scenarios where fast, real-time detection is a priority.

### **4.3. RetinaNet**

> [RetinaNet](https://arxiv.org/abs/1708.02002) is an object detection model that was developed to address the challenge of detecting objects across a range of scales. Traditional object detection models often struggle with the detection of small objects in an image, primarily due to the imbalance between the number of easy & hard examples during training. RetinaNet introduces a novel component called Focal Loss, designed to rectify this imbalance by focusing more on hard, or easily misclassified, examples. This focus enables the model to improve its performance on detecting challenging objects, which are typically small or less prominent within the image.

> The architecture of RetinaNet combines the strengths of two popular object detection paradigms: the speed & efficiency of single-stage detectors & the accuracy of two-stage detectors. It does this by using a [Feature Pyramid Network](https://www.youtube.com/watch?v=2-GGyW59CLw) (FPN) as a backbone on top of a ResNet architecture. The FPN effectively captures rich contextual information at different scales, making RetinaNet particularly adept at detecting objects of various sizes throughout an image. Coupled with the Focal Loss function, this architecture allows RetinaNet to achieve high levels of accuracy, rivaling more complex two-stage detectors, while still maintaining the efficiency & speed of single-stage detectors.

> However, despite its advancements, RetinaNet has some limitations. The model's increased focus on hard examples, while beneficial for small object detection, can sometimes lead to longer training times as the model works to converge. This is due to the added complexity introduced by the Focal Loss function. Additionally, while RetinaNet is more efficient than some two-stage models, it can still be computationally intensive compared to simpler single-stage detectors, particularly when dealing with very large images or high-resolution inputs. This computational demand can pose challenges in resource-constrained environments or applications requiring real-time processing. Despite these challenges, RetinaNet stands as a significant development in the field of object detection, offering a robust solution for detecting objects across a wide range of sizes with high accuracy.

### **4.4. DETR (End-to-End Object Detection with Transformers)**

> [DETR](https://arxiv.org/abs/2005.12872) (End-to-End Object Detection with Transformers) is a novel approach to object detection that incorporates the transformer architecture, traditionally used in natural language processing, into a vision-based task. This model represents a significant shift from conventional object detection methods that rely on region proposal networks & various post-processing steps. DETR aims to simplify the object detection pipeline by treating object detection as a direct set prediction problem. It eliminates the need for many hand-engineered components like anchor generation & non-maximum suppression, which are common in other object detection frameworks. The core idea is to use a transformer coupled with a Convolutional NN (CNN) to process the entire image & output a fixed-size set of predictions, including the class labels & bounding boxes of objects present.

> The architecture of DETR consists of a CNN backbone for feature extraction, followed by a transformer encoder-decoder structure. The CNN processes the input image & generates a feature map, which is then flattened & fed into the transformer. The transformer encoder-decoder mechanism, with its self-attention & cross-attention capabilities, is adept at capturing global dependencies within the data. This allows DETR to consider the entire context of the image when making predictions, leading to more accurate & coherent detection results, particularly in complex scenes with multiple interacting objects. The model outputs a set of predictions that are processed by a feed-forward network to produce the final bounding boxes & class labels.

> However, DETR also presents some challenges. One of the main drawbacks is its relatively slow convergence during training, primarily due to the complexity of the transformer architecture & the global nature of its feature processing. Training DETR to achieve competitive performance can take significantly longer than training traditional object detection models. Additionally, while DETR simplifies the object detection pipeline & reduces the reliance on hand-tuned components, it can struggle with detecting small objects. This is partly because global context captured by transformers might not always be sufficient to discern smaller, less prominent objects in an image. Despite these challenges, DETR's innovative use of transformers in object detection demonstrates a promising new direction in computer vision, potentially paving the way for more unified & simplified approaches to complex visual tasks.

## **5. Generative Models**

A generative model is a statistical model that is capable of generating new data points.


### **5.1. GANs (Generative Adversarial Networks)**

> [Generative Adversarial Networks](https://dl.acm.org/doi/abs/10.1145/3422622) (GANs) are a class of AInsupervised ML, implemented by a system of two NNs contesting with each other in a zero-sum game framework. Introduced by Ian Goodfellow & his colleagues in 2014, GANs consist of two parts: a generator & a discriminator. The generator creates data that is intended to pass for real data, while the discriminator evaluates data & tries to determine whether it is real or produced by the generator. This adversarial process drives the generator to produce increasingly realistic data over time. GANs are primarily used for generating realistic images, but they also have applications in various fields, including art, medicine, & science, for tasks such as image enhancement, super-resolution, & image-to-image translation.

> [StyleGAN](https://arxiv.org/abs/1812.04948) & [ProGAN](https://arxiv.org/abs/1710.10196) are advanced examples of GANs that demonstrate the capabilities of these networks in generating high-resolution, realistic images. ProGAN, or Progressive Growing of GANs, introduced by Nvidia researchers, is known for its method of progressively increasing the size of the generated images, starting from lower resolutions & adding layers to the networks as training progresses. This technique allows ProGAN to effectively learn the coarse-to-fine details of images, making it particularly effective in generating high-quality images. StyleGAN, another Nvidia innovation, builds upon ProGAN by introducing a novel generator architecture that can control the style of the generated output at different levels of granularity. It achieves this by modifying aspects of the generative process at different scales, which allows for unprecedented control over the style & content of the generated images, leading to highly realistic & customizable outputs.

> Despite their impressive capabilities, GANs, including StyleGAN & ProGAN, have certain limitations & challenges. One significant issue is the difficulty in training GANs; the adversarial training process can be unstable & often leads to mode collapse, where the generator produces limited varieties of samples. Additionally, GANs require substantial computational resources for training, particularly for high-resolution image generation tasks. There are also ethical considerations, especially concerning the creation of deepfakes or the use of GANs to generate misleading or harmful content. Despite these challenges, GANs, StyleGAN, & ProGAN continue to be at the forefront of research in generative models, pushing the boundaries of what is possible in realistic image generation & opening new possibilities in various fields where image generation plays a crucial role.

### **5.2. Diffusion Models**

> Diffusion models are a class of generative models that have gained prominence for their ability to produce high-quality, detailed samples, such as images & audio. These models work on the principle of gradually transforming data from a simple distribution (such as Gaussian noise) into a complex one, resembling the target data distribution. The process involves a forward diffusion process, which gradually adds noise to the data until it is entirely random, & a reverse process, where a model learns to reconstruct the original data from the noise. This reverse process is learned by training a NN to predict the noise that was added at each step, effectively teaching it to denoise data. The generative capability of diffusion models has been demonstrated to be highly effective, particularly in fields like image & audio synthesis, where they can produce results with remarkable detail & coherence.

> Diffusion models offer several advantages over other types of generative models, such as Generative Adversarial Networks (GANs) & [variational autoencoders](https://towardsdatascience.com/understanding-variational-autoencoders-vaes-f70510919f73) (VAEs). One of their key strengths is the quality of the generated samples, which often surpasses that of other generative models, especially in terms of realism & diversity. Unlike GANs, diffusion models do not suffer from mode collapse (where the model generates a limited variety of outputs), & they provide a more stable training process. Additionally, diffusion models have a theoretically grounded framework based on Markov chains, providing a clear understanding of their functioning & properties. This theoretical basis is a significant advantage in the context of research & development of new generative models.

> However, diffusion models also have notable drawbacks. One of the primary issues is the computational cost & time required for both training & sample generation. The process of adding & then reversing noise is computationally intensive, making these models less efficient than alternatives like GANs in terms of generation speed. This can be a significant limitation in applications where real-time generation is crucial. Moreover, the quality & effectiveness of diffusion models are heavily reliant on the amount & quality of training data, & they may require substantial datasets to produce high-quality results. Despite these challenges, diffusion models represent a significant advancement in the field of generative models, offering a robust & high-quality approach to data generation that is applicable to a wide range of domains.

### **5.3. Transformer Models**


> Transformer models, introduced in the seminal 2017 paper "[Attention Is All You Need](https://arxiv.org/abs/1706.03762)" by Vaswani et al. in 2017, represent a fundamental shift in natural language processing (NLP) & have since become a cornerstone in the field. Prior to transformers, recurrent NNs (RNNs) & convolutional NNs (CNNs) were the standard architectures for handling sequential data like text. However, these models had limitations, particularly in handling long-range dependencies within the text due to issues like vanishing gradients in RNNs & the locality of convolution operations in CNNs. The transformer model addressed these challenges by relying solely on a mechanism known as attention, specifically self-attention, which allows the model to weigh the importance of different parts of the input data, regardless of their positional distance from each other. This enables transformers to process entire sequences of data simultaneously & capture complex, long-range dependencies more effectively.

> One of the main advantages of transformer models is their parallelizability, which significantly reduces training times. Unlike RNNs that process data sequentially, transformers can handle entire sequences in parallel, making them highly efficient & scalable with the increasing availability of powerful computing resources. This efficiency has enabled the training of very large models on extensive datasets, leading to substantial improvements in tasks like language translation, text generation, & sentiment analysis. Additionally, the self-attention mechanism provides a more nuanced understanding of the relationships & context within the data, resulting in models that can generate more coherent & contextually appropriate outputs.

> However, transformer models are not without their drawbacks. One significant issue is their requirement for substantial computational resources, both in terms of memory & processing power. Training large transformer models often necessitates the use of specialized hardware, such as GPUs or TPUs, which can be costly & less accessible for smaller organizations or individual researchers. Additionally, the size & complexity of these models can lead to challenges in interpretation & transparency, making it difficult to understand how & why specific decisions or predictions are made. Despite these challenges, transformer models have dramatically advanced the field of NLP & have been adapted & expanded upon in numerous subsequent models, solidifying their status as a key innovation in ML & AI.

### **5.4. BERT**

> [BERT](https://arxiv.org/abs/1810.04805) (Bidirectional Encoder Representations from Transformers) is a transformative model in the field of natural language processing (NLP), introduced by researchers at Google in 2018. It represents a significant departure from previous models that processed text in a unidirectional manner, either from left-to-right or right-to-left. BERT, instead, utilizes a bidirectional approach, allowing the model to consider the context of a word based on all of its surroundings within a sentence, rather than just the words that precede it. This is achieved through the use of the transformer architecture, specifically the attention mechanism, which enables the model to weigh the importance of different words in a sentence. BERT's ability to understand the full context of a word significantly improves its performance on a wide range of NLP tasks, such as question answering, sentiment analysis, & language inference.

> BERT's innovation lies in its pre-training & fine-tuning stages. During pre-training, the model is trained on a large corpus of text, learning general language patterns & structures. This is done using two unsupervised tasks: masked language modeling, where random words in a sentence are masked & the model learns to predict them, & next sentence prediction, where the model learns to predict whether two given sentences are logically connected. The pre-trained model can then be fine-tuned with additional output layers for various specific NLP tasks, requiring much less data for high-quality performance on these downstream tasks. This pre-training & fine-tuning approach enables BERT to achieve state-of-the-art results on numerous benchmarks.

> However, BERT also has its limitations. One of the main challenges is its demand for significant computational resources for both pre-training & fine-tuning, which can be a barrier for those without access to powerful hardware. Additionally, while BERT has made substantial advancements in understanding language context, it does not inherently capture the complexities of human language, such as idiomatic or nuanced expressions. The model's size & complexity also make it difficult to deploy in resource-constrained environments, such as on mobile devices or in real-time applications. Despite these challenges, BERT's impact on the field of NLP has been profound, paving the way for a range of applications & inspiring numerous subsequent models & research projects.

### **5.5. GPTs**

> The [GPT](https://arxiv.org/abs/2305.10435) (Generative Pre-trained Transformer) is a series of language models developed by OpenAI, with the most notable being GPT-3, known for its large scale & wide-ranging capabilities. The GPT models are based on the transformer architecture, which primarily uses self-attention mechanisms to process input data. Unlike BERT, which is bidirectional, GPT models are unidirectional, meaning they generate text based on preceding context only. This architecture allows GPT models to excel in tasks like text generation, where they can produce coherent & contextually relevant paragraphs of text. GPT models are trained on a diverse range of internet text sources, enabling them to learn a wide variety of language styles, topics, & information. Once pre-trained, they can be fine-tuned on specific tasks with smaller datasets, making them versatile tools for a range of language processing tasks.

> One of the key strengths of GPT models, particularly GPT-3, is their ability to perform a wide range of language tasks without task-specific data handling or training. This makes them highly versatile & capable of handling tasks like translation, question-answering, summarization, & even creative writing with a high degree of proficiency. The size & scale of GPT-3, with its 175 billion parameters, allow it to understand & generate text with a level of sophistication that was previously unachievable, providing insights & capabilities that are valuable for both practical applications & research purposes.

> However, the GPT series, & GPT-3 in particular, come with significant drawbacks. The model's large size entails substantial computational resources for training & inference, which can be prohibitively expensive & environmentally impactful due to the energy required. This scale also makes it challenging to deploy GPT-3 in environments with limited computational capacity. Furthermore, the model's outputs can sometimes include biased or inaccurate information, reflecting biases present in the training data. Additionally, while GPT-3 can generate impressively coherent text, it doesn't possess true understanding or reasoning capability, which can lead to nonsensical or factually incorrect outputs in complex scenarios. Despite these challenges, GPT models represent a significant advancement in natural language processing, offering powerful tools for generating human-like text & advancing the field of AI.

## **6. Multi-Modal Nets**

> Multi-modal AI models represent a significant advancement in the field of AI by integrating & interpreting information from multiple different data types or modalities, such as text, images, audio, & video. Traditional AI models typically focus on a single modality, like text in natural language processing or images in computer vision. However, multi-modal models are designed to process & relate information across these different forms, aiming to mimic the human ability to understand & interact with the world through multiple senses simultaneously. This integration allows for a more comprehensive understanding of complex data, which can be particularly useful in tasks that require context from more than one type of input, such as image captioning, visual question answering, or cross-modal information retrieval.

> The main advantage of multi-modal AI models is their ability to capture a richer & more nuanced understanding of data than single-modality models. For example, in a task like image captioning, a multi-modal model can generate descriptions of images that are not only accurate in terms of visual content but also contextually appropriate based on additional text information. This capability is crucial in applications like assistive technology for visually impaired users, where accurate & context-aware image descriptions can significantly enhance user experience. Similarly, in tasks like sentiment analysis, combining textual data with vocal intonation from audio inputs can lead to more accurate interpretations than using text or audio alone.

> However, multi-modal AI models come with their own set of challenges. One of the primary difficulties is the integration of different types of data, which often have varying formats, resolutions, & semantic structures. This requires complex architecture & sophisticated algorithms to effectively combine & interpret these diverse data streams. Another challenge is the requirement for large & diverse datasets that include multiple modalities, which can be difficult to curate & annotate. Additionally, the increased complexity of these models often leads to higher computational demands for training & inference, making them resource-intensive. Despite these challenges, the development of multi-modal AI models is a crucial step towards creating more intelligent & versatile AI systems, capable of understanding & interacting with the world in a more human-like manner.

> Here are a few multi-modal NN models:
> * FusionNet: FusionNet is a model designed for multi-modal sentiment analysis, combining text & image information to predict sentiment or emotions.
> * M3: M3 (Multi-Modal Multi-Scale) is a multi-modal model that integrates information from text, visual, & acoustic data for tasks like emotion recognition in audiovisual data.
> * MUSE: MUSE (Modality-Specific Unsupervised SEgmentation) is a model that segments & clusters data across different modalities, enabling unsupervised learning in multi-modal scenarios.
> * HMTL: HMTL (Heterogeneous Multi-Task Learning) is a framework that performs multi-modal learning across tasks, where each task may involve different data modalities.
> * MM-DNN: MM-DNN (Multi-Modal Deep NN) is a general framework for multi-modal learning that can be applied to various tasks, including image-text matching & classification.

# <font color="#418FDE" size="10" uppercase>**C: ML Major Models**</font>
----


In this lecture, you learned to:
* Investigate and apply conventional, popular, and memory-efficient ML architectures.
* Describe segmentation, object detection, generative, and multi-modal ML models.

In the next lecture (lecture D), we will talk about TensorFlow config.